# 🛡️ Project 03: AI-Based Spam Email Detection System
### Phase 3 — Model Training (All 5 Classifiers)

**Objective:** Train and compare Multinomial Naive Bayes, Logistic Regression, Random Forest, SVM (linear kernel), and LSTM on the preprocessed dataset.

### Phase 3 Checklist
- [ ] Load preprocessed data from Phase 2
- [ ] Train Naive Bayes
- [ ] Train Logistic Regression
- [ ] Train Random Forest
- [ ] Train SVM (linear kernel)
- [ ] Train LSTM (advanced track)
- [ ] Save all trained models

In [5]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix,
                              classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('../models', exist_ok=True)
print('All imports successful')

All imports successful


## 📂 Step 1 — Load Preprocessed Data from Phase 2

In [6]:
X_train = joblib.load('../data/processed/X_train.pkl')
X_test  = joblib.load('../data/processed/X_test.pkl')
y_train = joblib.load('../data/processed/y_train.pkl')
y_test  = joblib.load('../data/processed/y_test.pkl')

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train spam ratio: {y_train.mean():.2%}')
print(f'y_test  spam ratio: {y_test.mean():.2%}')

X_train: (44, 92)
X_test : (11, 92)
y_train spam ratio: 45.45%
y_test  spam ratio: 45.45%


## 🔧 Step 2 — Helper: Evaluate & Save Model

A reusable function that prints the classification report and plots the confusion matrix for any trained model.

In [7]:
results = {}

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    cm   = confusion_matrix(y_test, y_pred)

    results[name] = {'Accuracy': acc, 'Precision': prec,
                     'Recall': rec, 'F1 Score': f1}

    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Ham','Spam']))

    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham','Spam'],
                yticklabels=['Ham','Spam'], ax=ax)
    ax.set_title(f'{name} — Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'../reports/cm_{name.replace(" ","_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Helper function ready')

Helper function ready


## 1️⃣ Model 1 — Multinomial Naive Bayes

A probabilistic classifier that works well with word frequency/TF-IDF features. Best for text classification with sparse matrices.

In [8]:
from sklearn.preprocessing import MaxAbsScaler
from scipy.sparse import csr_matrix

# Naive Bayes needs non-negative values — scale to [0,1]
mas = MaxAbsScaler()
X_train_nb = mas.fit_transform(X_train)
X_test_nb  = mas.transform(X_test)

nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_nb, y_train)

joblib.dump(nb_model, '../models/naive_bayes.pkl')
evaluate_model('Naive Bayes', nb_model, X_test_nb, y_test)

ValueError: Negative values in data passed to MultinomialNB (input X).